# TF-IDF + Logistic Regression — Hate Speech Detection

## 1. Imports

In [1]:
import sys
sys.path.append('..')

import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix, f1_score
from src.data.preprocessor import full_preprocess

## 2. Chargement des données

In [11]:
df = pd.read_csv("../data/raw/HateSpeechData.csv")
df = df.drop(columns=["Unnamed: 0"])
df.head()

,count,hate_speech,offensive_language,neither,class,tweet
0,3,0,0,3,2,!!! RT @mayasolovely: As a woman you shouldn't...
1,3,0,3,0,1,!!!!! RT @mleew17: boy dats cold...tyga dwn ba...
2,3,0,3,0,1,!!!!!!! RT @UrKindOfBrand Dawg!!!! RT @80sbaby...
3,3,0,2,1,1,!!!!!!!!! RT @C_G_Anderson: @viva_based she lo...
4,6,0,6,0,1,!!!!!!!!!!!!! RT @ShenikaRoberts: The shit you...


## 3. Préprocessing

In [12]:
df["text_clean"] = df["tweet"].apply(full_preprocess)
df[["tweet", "text_clean"]].head()

,tweet,text_clean
0,!!! RT @mayasolovely: As a woman you shouldn't...,woman shouldnt complain clean house. amp man t...
1,!!!!! RT @mleew17: boy dats cold...tyga dwn ba...,boy dats cold...tyga dwn bad cuffin dat hoe 1s...
2,!!!!!!! RT @UrKindOfBrand Dawg!!!! RT @80sbaby...,dawg fuck bitch start confuse shit
3,!!!!!!!!! RT @C_G_Anderson: @viva_based she lo...,look like tranny
4,!!!!!!!!!!!!! RT @ShenikaRoberts: The shit you...,shit hear true faker bitch tell ya


## 4. TF-IDF Vectorization

In [13]:
vectorizer = TfidfVectorizer(max_features=10000)
X = vectorizer.fit_transform(df["text_clean"])
y = df["class"]

## 5. Train/Test Split

In [14]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

## 6. Modèle Baseline (sans class_weight)

In [15]:
model_baseline = LogisticRegression(max_iter=1000)
model_baseline.fit(X_train, y_train)
y_pred_baseline = model_baseline.predict(X_test)

print("Classification Report — Baseline:")
print(classification_report(y_test, y_pred_baseline))
print("Macro F1:", f1_score(y_test, y_pred_baseline, average='macro'))
print("Confusion Matrix:\n", confusion_matrix(y_test, y_pred_baseline))

Classification Report — Baseline:
              precision    recall  f1-score   support

           0       0.62      0.14      0.23       286
           1       0.91      0.97      0.94      3838
           2       0.86      0.84      0.85       833

    accuracy                           0.90      4957
   macro avg       0.80      0.65      0.67      4957
weighted avg       0.89      0.90      0.89      4957

Macro F1: 0.6741384645045209
Confusion Matrix:
 [[  40  221   25]
 [  25 3727   86]
 [   0  131  702]]


## 7. Modèle Amélioré (class_weight='balanced')

In [16]:
model_balanced = LogisticRegression(max_iter=1000, class_weight='balanced')
model_balanced.fit(X_train, y_train)
y_pred_balanced = model_balanced.predict(X_test)

print("Classification Report — Balanced:")
print(classification_report(y_test, y_pred_balanced))
print("Macro F1:", f1_score(y_test, y_pred_balanced, average='macro'))
print("Confusion Matrix:\n", confusion_matrix(y_test, y_pred_balanced))

Classification Report — Balanced:
              precision    recall  f1-score   support

           0       0.30      0.59      0.39       286
           1       0.97      0.85      0.90      3838
           2       0.77      0.95      0.85       833

    accuracy                           0.85      4957
   macro avg       0.68      0.80      0.72      4957
weighted avg       0.90      0.85      0.87      4957

Macro F1: 0.7168534765553564
Confusion Matrix:
 [[ 168   86   32]
 [ 382 3255  201]
 [  16   25  792]]


## 8. SMOTE (Oversampling)

In [17]:
from imblearn.over_sampling import SMOTE

sm = SMOTE(random_state=42)
X_train_sm, y_train_sm = sm.fit_resample(X_train, y_train)

model_smote = LogisticRegression(max_iter=1000)
model_smote.fit(X_train_sm, y_train_sm)
y_pred_smote = model_smote.predict(X_test)

print("Classification Report — SMOTE:")
print(classification_report(y_test, y_pred_smote))
print("Macro F1:", f1_score(y_test, y_pred_smote, average='macro'))
print("Confusion Matrix:\n", confusion_matrix(y_test, y_pred_smote))

Classification Report — SMOTE:
              precision    recall  f1-score   support

           0       0.30      0.52      0.38       286
           1       0.96      0.87      0.91      3838
           2       0.77      0.92      0.84       833

    accuracy                           0.86      4957
   macro avg       0.68      0.77      0.71      4957
weighted avg       0.89      0.86      0.87      4957

Macro F1: 0.7108871222500567
Confusion Matrix:
 [[ 148   97   41]
 [ 328 3324  186]
 [  14   49  770]]


## 9. Conclusion — Comparaison des modèles

In [18]:
results = {
    "Baseline":  f1_score(y_test, y_pred_baseline, average='macro'),
    "Balanced":  f1_score(y_test, y_pred_balanced, average='macro'),
    "SMOTE":     f1_score(y_test, y_pred_smote,    average='macro'),
}

for name, score in results.items():
    print(f"{name:10s} → Macro F1: {score:.4f}")

best = max(results, key=results.get)
print(f"\n Meilleur modèle : {best}")

Baseline   → Macro F1: 0.6741
Balanced   → Macro F1: 0.7169
SMOTE      → Macro F1: 0.7109

 Meilleur modèle : Balanced
